# Qwen3-14B: simulator vs a real A100

The same comparison as `A100_Validation_Colab.ipynb`, with **Qwen3-14B** instead of GPT-2: the same nine traffic patterns (scaled down, because a 14B model is about 100× slower per token), run on the real GPU and in the simulator with identical requests. Qwen3-14B is ungated, so no Hugging Face token is needed.

| line in the plots | what it is |
|---|---|
| **real A100** (black) | our scheduler drives real Qwen3-14B (bf16, eager attention), one forward pass per request per step, while NVML records power |
| **simulator** (yellow) | `run_engine` on its own clock, pricing Qwen3's kernels with EnergAIzer |

**Where the Qwen3 kernels come from.** EnergAIzer ships no Qwen3 trace, and tracing a 14B model at every anchor shape would not fit. A kernel's shape depends only on the architecture, so `dynshape.llama` writes the list directly: every kernel HuggingFace's eager `Qwen3ForCausalLM` launches, predicted from transformers' source as **2,628 kernels per prefill pass and 2,588 per decode step**. Qwen3 is LLaMA-shaped with three differences the list includes: an RMSNorm on the queries and keys of every layer (QK-norm), 8 key/value heads shared by 40 query heads (eager attention copies K and V out to all 40), and a 151,936-word vocabulary, so a much larger LM head. Section 5 checks the list against the profiler before anything is simulated.

| section | what it does |
|---|---|
| 1 | setup and the unit tests for the model code |
| 2-3 | lock the clock at 900 MHz, measure idle, load Qwen3-14B (29.5 GB) |
| 4 | build EnergAIzer and the Qwen3 templates |
| 5 | **kernel check**: real vs predicted kernels, op by op, with GPU time per op |
| 6 | **one pass at a time**: measured energy per pass vs the simulator's |
| 7-11 | the traffic runs: real, CPU-time fit, idle floor, simulator, plots, export |

**Needs an A100 runtime** (Runtime → Change runtime type → A100), ideally with High-RAM. About 45-60 minutes: 5-10 to download the weights, 3 to build EnergAIzer, about 5 of real runs, and the rest pricing new kernel shapes.

## 0 · Settings

The nine patterns keep their shape but are much smaller: a 14B model takes about 20-30 ms per decode pass and about 0.2 s per 1,000-token prefill on this card, against about 2 ms and 11 ms for GPT-2. The 29.5 GB of weights leave about 10 GB on a 40 GB card; with 8 key/value heads the KV cache is only 0.16 MB per token, so memory is not the limit here.

`KV_FRACTION` is the share of free GPU memory the scheduler may use for the KV cache; the rest is headroom for activations (the LM head's logits alone are 0.3 GB for a 1,000-token prompt).

In [ ]:
MODEL_ID   = 'Qwen/Qwen3-14B'   # ungated: no token needed

CONDITIONS = {
    # Mixed: zipf lengths, prompt:output 4:1, prompt + output <= 960 tokens.
    'light':  dict(kind='gamma', n=16, qps=0.5, cv=1.0),     # requests rarely overlap
    'bursty': dict(kind='gamma', n=16, qps=50.0, cv=2.0),    # all arrive at once: a decode backlog
    'spiky':  dict(kind='bursts', bursts=4, per_burst=6, spacing_s=8.0,
                   prompt=896, decode=16),                    # walls of long prompts, 8 s apart
}
RUN        = ['light', 'bursty', 'spiky']
REPEATS    = 1      # real runs per condition
BUCKET     = 32     # simulator: decode contexts rounded to multiples of this (1 = exact, slow)
EXTRAS     = True   # simulator: the kernels the template leaves out (KV cats, mask, embedding, copies)
HOST_TIME  = True   # simulator: charge the CPU time per step, fitted from the real runs (8b)
HOST_WAIT_W = 0.5   # GPU power above idle while it waits for the CPU (measured on GPT-2: +0.49 W)
KV_FRACTION = 0.6   # share of free GPU memory after loading the model (about 10 GB) for the KV cache
PASS_SHAPES = [('prefill 1000', 'prefill', 1000, 0), ('prefill 128', 'prefill', 128, 0),
               ('decode @128', 'decode', 1, 128), ('decode @1000', 'decode', 1, 1000)]
LOOP_S     = 15.0   # seconds of back-to-back passes per shape in section 6
BRANCH     = 'extra-kernels'   # GitHub branch with dynshape.llama (Qwen3 support)
TARGET_MHZ = 900    # the clock EnergAIzer's tables were measured at
PLOT_DT_S  = 1.0    # window for the plots and the comparison
MAX_RUN_S  = 600    # safety stop for one real run

## 1 · Setup

In [ ]:
import os, subprocess, sys, time, json, contextlib, dataclasses, collections
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')   # the KV cats reallocate every step

REPO = 'https://github.com/shubhamOjha1000/dynamic_shape_power_sim.git'
DIR  = '/content/dynamic_shape_power_sim'
if not os.path.isdir(DIR):
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO, DIR], check=True)
else:
    subprocess.run(['git', '-C', DIR, 'fetch', '--depth', '1', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', DIR, 'checkout', '-f', '-B', BRANCH, 'FETCH_HEAD'], check=True)
if DIR not in sys.path:
    sys.path.insert(0, DIR)
os.chdir(DIR)
head = subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout.strip()
print(f'{BRANCH}: {head}')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'nvidia-ml-py', 'transformers>=4.44,<5', 'accelerate', 'huggingface_hub', 'pytest',
                'pyyaml', 'scipy', 'scikit-learn', 'cvxpy', 'opt-einsum', 'tqdm', 'gdown'],
               check=True)

import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch, transformers, dynshape
from dynshape import LlamaShape, llama_extras

HF_TOKEN = os.environ.get('HF_TOKEN')
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN') or HF_TOKEN
except Exception as exc:
    print(f'no HF_TOKEN Colab secret ({type(exc).__name__}) -- not needed for Qwen3-14B')

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)
print('dynshape', dynshape.__version__, '| torch', torch.__version__,
      '| transformers', transformers.__version__, '| HF token:', 'yes' if HF_TOKEN else 'no')
assert torch.cuda.is_available(), 'no GPU -- Runtime > Change runtime type > A100'
print('device  ', torch.cuda.get_device_name(0))

In [ ]:
# The LLaMA template code's own unit tests (CPU, about a minute), before the long run.
t0 = time.time()
r = subprocess.run([sys.executable, '-m', 'pytest', '-q', '--tb=short', '-rf',
                    '-p', 'no:cacheprovider', 'tests/test_llama.py', 'tests/test_extras.py'],
                   capture_output=True, text=True)
for line in (r.stdout + r.stderr).strip().splitlines()[-30:]:
    print(' ', line)
verdict = 'all passed' if r.returncode == 0 else 'SOME FAILED -- the rest still runs; send this output'
print(f'unit tests: {verdict}   [{time.time() - t0:.0f} s]')

## 2 · Lock the clock, and measure this card's idle floor

EnergAIzer's tables were measured at 900 MHz, so the card is locked there; persistence mode keeps the lock while nothing runs. The idle floor is measured rather than assumed, because Colab's card is an A100-**SXM4** and the tables come from an A100-**PCIe**.

The sampler records board power, NVML's cumulative energy counter and the SM clock every 10 ms.

In [ ]:
import threading, pynvml

def sh(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    return r.returncode == 0, (r.stdout + r.stderr).strip()

for cmd in (['nvidia-smi', '-pm', '1'], ['nvidia-smi', '-lgc', f'{TARGET_MHZ},{TARGET_MHZ}']):
    ok, msg = sh(cmd)
    print(' '.join(cmd), '->', 'ok' if ok else msg[:150])

pynvml.nvmlInit()
HANDLE = pynvml.nvmlDeviceGetHandleByIndex(0)
CARD = pynvml.nvmlDeviceGetName(HANDLE)
CARD = CARD.decode() if isinstance(CARD, bytes) else CARD

def _energy_j():
    return pynvml.nvmlDeviceGetTotalEnergyConsumption(HANDLE) / 1000.0     # mJ -> J

try:
    _energy_j()
    HAVE_ENERGY = True
except pynvml.NVMLError:
    HAVE_ENERGY = False


class NvmlSampler:
    '''Board power, the cumulative energy counter and the SM clock, every interval_s.'''

    def __init__(self, interval_s=0.01):
        self.interval_s = interval_s
        self._stop = threading.Event()

    def _loop(self):
        while not self._stop.is_set():
            try:
                t = time.perf_counter() - self.t0
                p = pynvml.nvmlDeviceGetPowerUsage(HANDLE) / 1000.0
                e = _energy_j() if HAVE_ENERGY else np.nan
                c = pynvml.nvmlDeviceGetClockInfo(HANDLE, pynvml.NVML_CLOCK_SM)
                self.rows.append((t, p, e, c))
            except pynvml.NVMLError:
                self.errors += 1
            self._stop.wait(self.interval_s)

    def start(self):
        self.rows, self.errors = [], 0
        self._stop.clear()
        self.t0 = time.perf_counter()
        self._thread = threading.Thread(target=self._loop, daemon=True)
        self._thread.start()
        return self

    def stop(self):
        self._stop.set()
        self._thread.join(2.0)
        df = pd.DataFrame(self.rows, columns=['t_s', 'power_w', 'energy_j', 'sm_mhz'])
        df['energy_j'] -= df['energy_j'].iloc[0]
        return df


time.sleep(2)
counter_msg = 'yes' if HAVE_ENERGY else 'NOT SUPPORTED -- energy comes from integrating power'
print()
print(f'card           : {CARD}')
print(f'energy counter : {counter_msg}')

idle_df = NvmlSampler(0.02).start()
time.sleep(30)
idle_df = idle_df.stop()
IDLE_W = float(idle_df.power_w.median())
print(f'idle at {TARGET_MHZ} MHz : {IDLE_W:.2f} W median over 30 s, '
      f'clock {idle_df.sm_mhz.median():.0f} MHz')
print(f'EnergAIzer tables (A100-PCIe, 900 MHz): 47.35 W, difference {IDLE_W - 47.35:+.2f} W')

# Does the lock hold under load? A matmul loop for 3 s.
x = torch.randn(4096, 4096, device='cuda', dtype=torch.bfloat16)
load_df = NvmlSampler(0.02).start()
t_end = time.time() + 3
while time.time() < t_end:
    for _ in range(20):
        x @ x
    torch.cuda.synchronize()
load_df = load_df.stop()
del x
LOAD_MHZ = float(load_df.sm_mhz.median())
print(f'under load     : {LOAD_MHZ:.0f} MHz median, {load_df.power_w.median():.0f} W')
if abs(LOAD_MHZ - TARGET_MHZ) > 15:
    print('WARNING: the clock did not hold under load -- compare shapes, not absolute watts.')

## 3 · Qwen3-14B, and the executor

bf16 weights (29.5 GB) with **eager** attention, the same executor as for GPT-2: **one forward pass per request per step**, each request with its own KV cache, which is what the simulator prices with `fuse_linear=False`. Prompts are random token ids, so outputs are meaningless: this is a power benchmark.

The KV-cache budget the scheduler gets is `KV_FRACTION` of the memory still free after loading, at 0.16 MB per token.

In [ ]:
from transformers import AutoModelForCausalLM, DynamicCache
from dynshape import prompt_token_ids

DEV = 'cuda'
t0 = time.time()
try:
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16,
                                                 attn_implementation='eager',
                                                 device_map=DEV, token=HF_TOKEN).eval()
except Exception as exc:
    raise RuntimeError(
        f'could not load {MODEL_ID}: {type(exc).__name__}: {str(exc)[:300]}\n'
        'Qwen3 needs transformers >= 4.51; a gated MODEL_ID would also need an HF_TOKEN '
        'Colab secret (key icon on the left) with notebook access on.') from exc
torch.set_grad_enabled(False)
SHAPE = LlamaShape.from_hf_config(model.config, name=MODEL_ID.split('/')[-1].lower())
MC = SHAPE.model_config()
print(f'{MODEL_ID} loaded in {time.time() - t0:.0f} s')
print(f'  {SHAPE}')
print(f'  attention: {model.config._attn_implementation}, {SHAPE.parameters() / 1e9:.1f} B parameters')

PROMPT = torch.randint(3, SHAPE.vocab, (1, SHAPE.max_position), device=DEV)
for _ in range(2):                  # cuBLAS handles, autotuning, allocator growth
    model(input_ids=PROMPT[:, :128])
torch.cuda.synchronize()

free_b, total_b = torch.cuda.mem_get_info()
KV_TOKENS = int(KV_FRACTION * free_b / SHAPE.kv_bytes_per_token())
KV_BLOCKS = max(1, KV_TOKENS // 16)
print(f'GPU memory: {torch.cuda.memory_allocated() / 1e9:.1f} GB in use, '
      f'{free_b / 1e9:.1f} GB free of {total_b / 1e9:.1f} GB')
print(f'KV cache budget: {KV_TOKENS:,} tokens ({SHAPE.kv_bytes_per_token() / 1e6:.2f} MB each) '
      f'-> {KV_BLOCKS:,} blocks of 16')


def gpu_sync():
    torch.cuda.synchronize()


class Executor:
    '''One real forward pass per Piece, each request holding its own KV cache.'''

    def __init__(self):
        self.cache, self.prompt, self.next_token = {}, {}, {}

    def admit(self, rid, n_prompt):
        self.prompt[rid] = torch.tensor(prompt_token_ids(n_prompt, seed=rid, vocab=SHAPE.vocab),
                                        dtype=torch.long, device=DEV)

    def release(self, rid):
        for d in (self.cache, self.prompt, self.next_token):
            d.pop(rid, None)

    def run_piece(self, piece):
        rid = piece.request_id
        if piece.kind == 'prefill':
            if piece.context == 0:                  # fresh start, or a restart after preemption
                self.cache[rid] = DynamicCache()
            ids = self.prompt[rid][piece.context: piece.context + piece.tokens][None, :]
        else:
            ids = self.next_token[rid].view(1, 1)
        out = model(input_ids=ids, past_key_values=self.cache[rid], use_cache=True)
        self.cache[rid] = out.past_key_values
        self.next_token[rid] = out.logits[0, -1].argmax()


def exec_pass(kind, tokens, ctx):
    '''One forward pass exactly as the executor runs it, repeatable: a decode pass crops
    its cache back after every call.'''
    if kind == 'prefill':
        ids = PROMPT[:, :tokens]
        return lambda: model(input_ids=ids, past_key_values=DynamicCache(), use_cache=True)
    cache = model(input_ids=PROMPT[:, :ctx], use_cache=True).past_key_values
    ids = PROMPT[:, ctx:ctx + 1]

    def fn():
        model(input_ids=ids, past_key_values=cache, use_cache=True)
        cache.crop(ctx)
    return fn

print('executor ready')

## 4 · EnergAIzer, and the Qwen3 templates

The same EnergAIzer build as for GPT-2. Then `write_llama_templates` writes three prefill anchors for this model's shape, and `ShapeRewriter` learns their scaling law exactly as it does from the GPT-2 traces. Decode shapes come from the same law through the query/key split, so no decode trace is needed. The classifier report should say 40 attention runs (one per layer), and the extras line 210 and 170 extra kernels per prefill pass and decode step.

In [ ]:
from dynshape import ShapeRewriter, write_llama_templates
from dynshape.llama import ANCHORS, n_template_entries
from dynshape.mixed import mixed_report, extras_report
from dynshape.energaizer import (clone_artifact, download_lut, lut_status, locate_artifact,
                                 build_gee_predictor, measured_idle_power_w)

TDIR = f'templates/{SHAPE.name}'
write_llama_templates(SHAPE, TDIR)
rw = ShapeRewriter.from_dir(TDIR, model=SHAPE.name, b0=ANCHORS[0][0], s0=ANCHORS[0][1])
XS = llama_extras(SHAPE) if EXTRAS else None
print(f'{SHAPE.name}: {len(rw.base)} template entries (predicted {n_template_entries(SHAPE)})')
print(' ', mixed_report(rw))
if XS is not None:
    print(' ', extras_report(rw, XS))
BENCH = pd.DataFrame()               # no measured prices for Qwen3 yet

ART = clone_artifact('/content/energaizer')
download_lut(ART)
st = lut_status(locate_artifact(ART))
assert st['complete'], 'LUT incomplete: ' + str(st['missing'])
t0 = time.time()
gee = build_gee_predictor(ART)
LUT_IDLE_W = measured_idle_power_w(ART, 900)      # the floor inside every predicted kernel
print(f'EnergAIzer built in {time.time() - t0:.0f} s; idle floor {LUT_IDLE_W:.2f} W')

In [ ]:
from contextlib import contextmanager
import dynshape.mixed as mixed
from dynshape import Piece
from dynshape.mixed import build_iteration_kernels_tagged

# The original, exact expanders -- kept so rerunning this cell never wraps a wrapper.
# `_piece_extras` sizes the extra kernels; it is None on a checkout without them.
_exact_piece_entries = getattr(mixed, '_exact_piece_entries', mixed._piece_entries)
mixed._exact_piece_entries = _exact_piece_entries
_exact_piece_extras = getattr(mixed, '_exact_piece_extras', getattr(mixed, '_piece_extras', None))
mixed._exact_piece_extras = _exact_piece_extras

def bucket_context(ctx, size=None):
    '''Round a decode context to the nearest multiple of size, never below size.'''
    size = BUCKET if size is None else size
    return max(size, int(round(ctx / size)) * size)

def _bucketed(piece):
    if piece.kind == 'decode' and BUCKET > 1:
        return dataclasses.replace(piece, context=bucket_context(piece.context))
    return piece

def _bucketed_piece_entries(rewriter, piece, indices=None):
    return _exact_piece_entries(rewriter, _bucketed(piece), indices)

def _bucketed_piece_extras(extras, piece):
    return _exact_piece_extras(extras, _bucketed(piece))

def set_bucketing(on):
    '''Bucket decode contexts for the template and for the extra kernels, or for neither.'''
    mixed._piece_entries = _bucketed_piece_entries if on else _exact_piece_entries
    if _exact_piece_extras is not None:
        mixed._piece_extras = _bucketed_piece_extras if on else _exact_piece_extras

set_bucketing(BUCKET > 1)

@contextmanager
def exact_contexts():
    '''Temporarily switch bucketing off -- for the shapes the GPU really ran.'''
    saved = (mixed._piece_entries, getattr(mixed, '_piece_extras', None))
    set_bucketing(False)
    try:
        yield
    finally:
        mixed._piece_entries = saved[0]
        if saved[1] is not None:
            mixed._piece_extras = saved[1]

print(f'BUCKET = {BUCKET}' + ('  (exact contexts)' if BUCKET <= 1 else ''))

## 5 · Kernel check: does the simulator launch what the GPU launches?

The profiler records every op that launched GPU work in a real 512-token prefill and a real decode step at context 512. Each is matched, op by op, against the simulator's list for the same pass: the generated template plus the extras, each entry labelled with the op it stands for.

- **`diff` = 0 on every row** means the predicted kernel list is complete for this transformers version. A non-zero row names the op that is missing or extra.
- **`real_gpu_ms` vs `energaizer_ms`** per op shows where EnergAIzer's timing is off, the way GPT-2's LM head was 3.5x off. Watch `aten::mm`: it includes Qwen3's 151,936-wide LM head.
- `real_kernels` can exceed `real_ops` when cuBLAS splits a GEMM into two kernels; the simulator prices such a GEMM as one.

In [ ]:
from torch.profiler import profile, ProfilerActivity
from dynshape import Piece
from dynshape.mixed import build_iteration_kernels_tagged
from dynshape.llama import llama_template_labeled

ACTIVE = 3
CHECK = [('prefill 512', 'prefill', 512, 0), ('decode @512', 'decode', 1, 512)]
TEMPLATE_LABELS = [lab for _e, lab in llama_template_labeled(SHAPE, 1, 128)]
EXTRA_OP = {'embed_lookup': 'aten::index_select', 'arange_q': 'aten::arange',
            'arange_k': 'aten::arange', 'arange_scalar': 'aten::arange',
            'mask_offset': 'aten::add_', 'mask_compare': 'aten::le',
            'mask_scalar_copy': 'aten::copy_', 'mask_fill': 'aten::fill_',
            'mask_where': 'aten::where', 'kv_cache_cat': 'aten::cat',
            'merge_heads_copy': 'aten::copy_', 'repeat_kv': 'aten::copy_'}
UNPRICED = collections.Counter()


def profile_ops(fn, warmup=2):
    '''(per-op table, kernels per pass, GPU ms per pass) over ACTIVE passes.'''
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA]) as prof:
        for _ in range(ACTIVE):
            fn()
        torch.cuda.synchronize()
    rows = []
    for ev in prof.events():
        ks = getattr(ev, 'kernels', None) or []
        if ks:
            rows.append(dict(op=ev.name, kernels=len(ks),
                             gpu_ms=sum(float(k.duration) for k in ks) / 1000.0))
    prof.export_chrome_trace('/content/model_trace.json')
    traced = [e for e in json.load(open('/content/model_trace.json'))['traceEvents']
              if e.get('cat') in ('kernel', 'gpu_memcpy', 'gpu_memset')]
    return (pd.DataFrame(rows, columns=['op', 'kernels', 'gpu_ms']), len(traced) / ACTIVE,
            sum(e['dur'] for e in traced) / ACTIVE / 1000.0)


def piece_of(kind, tokens, ctx):
    return Piece(kind=kind, request_id=0, tokens=tokens, context=ctx)


def sim_ops(piece):
    '''The simulator's kernels for one pass: the op each stands for, EnergAIzer's time
    and its energy above EnergAIzer's idle floor.'''
    rows, i = [], 0
    for (q, op), tag in build_iteration_kernels_tagged(rw, [piece], fuse_linear=False, extras=XS):
        name = tag.partition('+')[2]
        if name:
            label = EXTRA_OP.get(name, name)
        else:
            label = TEMPLATE_LABELS[i]
            i += 1
        try:
            t_ms, _p, e_j = gee.predict(q, op)
            dyn = 1000.0 * e_j - LUT_IDLE_W * t_ms
        except Exception as exc:
            UNPRICED[(op[0], type(exc).__name__)] += 1
            t_ms = dyn = np.nan
        rows.append(dict(op=label, entry=name or op[0], ms=t_ms, dyn_mj=dyn))
    return pd.DataFrame(rows)


OPCHECK, KTOTALS = [], []
for label, kind, tokens, ctx in CHECK:
    t0 = time.time()
    fn = exec_pass(kind, tokens, ctx)
    real, n_kernels, gpu_ms = profile_ops(fn)
    del fn
    r = real.groupby('op').agg(real_ops=('kernels', 'size'), real_kernels=('kernels', 'sum'),
                               real_gpu_ms=('gpu_ms', 'sum')) / ACTIVE
    s = sim_ops(piece_of(kind, tokens, ctx))
    sg = s.groupby('op').agg(sim_entries=('ms', 'size'), energaizer_ms=('ms', 'sum'),
                             sim_dyn_mj=('dyn_mj', 'sum'))
    t = r.join(sg, how='outer').fillna(0.0)
    t['diff'] = t.real_ops - t.sim_entries
    t = t.sort_values('real_gpu_ms', ascending=False)
    OPCHECK.append(t.reset_index().assign(case=label))
    KTOTALS.append(dict(case=label, real_ops=float(t.real_ops.sum()), real_kernels=n_kernels,
                        sim_entries=len(s), real_gpu_ms=gpu_ms, energaizer_ms=float(s.ms.sum()),
                        sim_dyn_mj=float(s.dyn_mj.sum())))
    bad = t[t['diff'].abs() > 0.01]
    verdict = 'every op matches' if bad.empty else f'{len(bad)} ops differ: ' + ', '.join(
        f'{op} ({d:+.0f})' for op, d in bad['diff'].items())
    print(f'{label}: real {n_kernels:.0f} kernels from {t.real_ops.sum():.0f} ops, '
          f'{gpu_ms:.2f} GPU ms | simulator {len(s)} entries, {s.ms.sum():.2f} EnergAIzer ms '
          f'| {verdict}   [{time.time() - t0:.0f} s]')
    display(t.round(3))
OPCHECK = pd.concat(OPCHECK, ignore_index=True)
KTOTALS = pd.DataFrame(KTOTALS).set_index('case')
display(KTOTALS.round(2))
unpriced_msg = dict(UNPRICED) or 'none'
print(f'entries EnergAIzer could not price: {unpriced_msg}')

## 6 · One pass at a time: measured energy vs the simulator

Each shape in `PASS_SHAPES` runs back to back for `LOOP_S` seconds while NVML's energy counter runs, giving the real energy per pass above idle. The CPU-waiting part (`HOST_WAIT_W` × the time the GPU is not busy) is taken off to get the energy of the GPU work, which is what the simulator's kernels should add up to.

A 14B model draws far more power above idle than GPT-2 (expect tens to hundreds of watts, against 3-4 W), so these numbers are much less sensitive to the idle floor.

In [ ]:
def _cum(nv):
    '''(t, E) on the sampler's own clock, keeping only the moments the counter changed.'''
    t = nv.t_s.to_numpy()
    if HAVE_ENERGY:
        e = nv.energy_j.to_numpy()
        keep = np.r_[True, np.diff(e) != 0]
        return t[keep], e[keep]
    w = nv.power_w.to_numpy()
    return t, np.r_[0.0, np.cumsum(0.5 * (w[1:] + w[:-1]) * np.diff(t))]


def steady(fn, seconds=LOOP_S, warm_s=3.0):
    '''Run fn back to back, synchronising after each pass; energy from NVML's counter.'''
    t_warm = time.perf_counter() + warm_s
    while time.perf_counter() < t_warm:
        fn()
        torch.cuda.synchronize()
    smp = NvmlSampler(0.01).start()
    time.sleep(0.3)
    start, n = time.perf_counter(), 0
    while time.perf_counter() - start < seconds:
        fn()
        torch.cuda.synchronize()
        n += 1
    stop = time.perf_counter()
    time.sleep(0.3)
    nv = smp.stop()
    a, b = start - smp.t0, stop - smp.t0
    t, e = _cum(nv)
    joules = float(np.interp(b, t, e) - np.interp(a, t, e))
    mid = nv[(nv.t_s >= a) & (nv.t_s <= b)]
    return dict(passes=n, ms_per_pass=1000.0 * (b - a) / n, board_mj_per_pass=1000.0 * joules / n,
                sm_mhz=float(mid.sm_mhz.median()))


rows = []
for label, kind, tokens, ctx in PASS_SHAPES:
    fn = exec_pass(kind, tokens, ctx)
    _real, _n, gpu_ms = profile_ops(fn)
    r = steady(fn)
    del fn
    torch.cuda.empty_cache()
    s = sim_ops(piece_of(kind, tokens, ctx))
    measured = r['board_mj_per_pass'] - IDLE_W * r['ms_per_pass']
    waiting = HOST_WAIT_W * max(0.0, r['ms_per_pass'] - gpu_ms)
    rows.append(dict(case=label, passes=r['passes'], ms_per_pass=r['ms_per_pass'],
                     real_gpu_ms=gpu_ms, energaizer_ms=float(s.ms.sum()),
                     measured_mj=measured, waiting_mj=waiting, gpu_work_mj=measured - waiting,
                     sim_mj=float(s.dyn_mj.sum()), w_above_idle=measured / r['ms_per_pass'],
                     sm_mhz=r['sm_mhz']))
    print(f"{label:13s} {r['ms_per_pass']:8.2f} ms/pass ({gpu_ms:7.2f} GPU ms) | real GPU work "
          f"{measured - waiting:9.1f} mJ | simulator {s.dyn_mj.sum():9.1f} mJ "
          f"({100 * (s.dyn_mj.sum() / (measured - waiting) - 1):+.0f}%) | {r['sm_mhz']:.0f} MHz")
PASSCHECK = pd.DataFrame(rows).set_index('case')
PASSCHECK['err_pct'] = 100 * (PASSCHECK.sim_mj / PASSCHECK.gpu_work_mj - 1)
PASSCHECK['time_ratio_real_over_energaizer'] = PASSCHECK.real_gpu_ms / PASSCHECK.energaizer_ms
display(PASSCHECK.round(2))

## 7 · The traffic

The nine patterns of the GPT-2 notebook, scaled for a 14B model. Every run, real or simulated, regenerates its requests from the same seed. The scheduler's KV pool is sized from the memory measured free in section 3 (`KV_BLOCKS`), so the scheduler preempts when the real GPU would run out.

In [ ]:
from dynshape import (TrafficConfig, generate_traffic, TraceInterval, FixedLength,
                      build_replay_traffic, traffic_summary, check_fits_context,
                      SchedulerConfig)

def make_traffic(name, seed=0):
    '''Fresh request objects for one condition. The scheduler mutates them, so every
    run, real or simulated, gets its own copy from the same seed.'''
    c = CONDITIONS[name]
    if 'prompt' not in c:                     # zipf lengths, as in the simulator notebook
        return build_replay_traffic(num_requests=c['n'], qps=c['qps'], cv=c['cv'],
                                    min_tokens=64, max_total_tokens=960, theta=0.85,
                                    prefill_to_decode_ratio=4.0, seed=seed)
    lengths = FixedLength(c['prompt'], c['decode'])
    if c['kind'] == 'gamma':
        cfg = TrafficConfig(interval='gamma', qps=c['qps'], cv=c['cv'],
                            length_generator=lengths, num_requests=c['n'], seed=seed)
    else:
        rng = np.random.default_rng(seed)
        times = sorted(t for b in range(c['bursts'])
                       for t in b * c['spacing_s'] + rng.uniform(0, 0.005, c['per_burst']))
        cfg = TrafficConfig(interval_generator=TraceInterval(times), length_generator=lengths,
                            num_requests=len(times), seed=seed)
    reqs = generate_traffic(cfg)
    check_fits_context(reqs, SHAPE.max_position)
    return reqs

def sched_for(name):
    '''One scheduler config per condition, shared by BOTH sides. With fixed-length
    prompts the token budget is a whole number of prompts, so a prompt is never split
    into small chunks -- every prefill kernel is full size.'''
    c = CONDITIONS[name]
    if 'prompt' not in c:
        chunk = 2048                                          # the simulator notebook's setting
    elif c['kind'] == 'bursts':
        chunk = c['per_burst'] * c['prompt'] + (512 if c['decode'] > 1 else 0)   # a whole wall per step
    else:
        chunk = 4 * c['prompt'] if c['decode'] <= 1 else max(4 * c['prompt'], 512)
        # up to 4 whole prompts per step; decode-only traffic needs room for its decodes
    return SchedulerConfig(chunk_size=chunk, max_num_seqs=256, block_size=16, max_tokens=2048,
                           num_blocks=globals().get('KV_BLOCKS'))   # KV pool sized from free GPU memory

TRAFFIC = pd.DataFrame({name: traffic_summary(make_traffic(name)) for name in RUN})
display(TRAFFIC.round(2))
for name in RUN:
    print(f'{name:15s} token budget per step: {sched_for(name).chunk_size:,}')

## 8 · The real runs

Our scheduler, untouched, on the real clock. Each step's pieces run as real forward passes, then `torch.cuda.synchronize()` makes the timing real. Every step records its start, its synchronized end, the exact pieces it ran and the scheduler's queue and KV state. Waits for the next arrival are recorded as idle.

**Energy from NVML.** On an A100 the energy counter only updates about every 100 ms. The helpers below keep just the moments it changed, so interpolating between them spreads each update over the time it actually covers. The last validation run interpolated between every 10 ms sample instead, which made 0.25 s windows flip between +14 W and -7 W.

In [ ]:
from dynshape import ChunkedPrefillScheduler, ModelConfig, HardwareConfig

def run_real(requests, sched_cfg, executor, sample_s=0.01, pad_s=1.0):
    '''Our scheduler on the real clock: one forward pass per piece, NVML alongside.'''
    sched = ChunkedPrefillScheduler(sched_cfg, MC, HardwareConfig())
    pending = sorted(requests, key=lambda r: (r.arrived_at, r.id))
    for r in pending:
        executor.admit(r.id, r.num_prefill_tokens)

    sampler = NvmlSampler(sample_s).start()
    time.sleep(pad_s)                              # a second of idle before the first request
    t0 = time.perf_counter()
    work_start_s = t0 - sampler.t0                 # traffic time 0, on the sampler clock

    steps, idles = [], []
    cursor = done = 0
    truncated = False
    while done < len(pending):
        now = time.perf_counter() - t0
        if now > MAX_RUN_S:
            truncated = True
            break
        while cursor < len(pending) and pending[cursor].arrived_at <= now:
            sched.add_request(pending[cursor])
            cursor += 1

        batch = sched.get_next_batch()
        if batch is None:
            if cursor >= len(pending):
                raise RuntimeError('nothing runnable and nothing left to arrive')
            a = time.perf_counter() - t0
            time.sleep(max(0.0, pending[cursor].arrived_at - a))
            idles.append((a, time.perf_counter() - t0))
            continue

        pieces = batch.pieces
        t_it = time.perf_counter() - t0
        batch.on_schedule(t_it)
        for p in pieces:
            executor.run_piece(p)
        gpu_sync()
        t_end = time.perf_counter() - t0
        steps.append(dict(t_start_s=t_it, t_end_s=t_end,
                          pieces=[(p.kind, p.request_id, p.tokens, p.context) for p in pieces],
                          n_waiting=sched.num_waiting, n_running=sched.num_running,
                          kv_blocks=sched.allocator.num_allocated_blocks,
                          kv_utilisation=sched.allocator.utilisation,
                          preemptions_total=sched.num_preemptions))

        batch.on_batch_end(time.perf_counter() - t0)
        sched.on_batch_end(batch)
        for r in batch.requests:
            if r.completed:
                executor.release(r.id)
                done += 1

    work_end_s = time.perf_counter() - t0
    time.sleep(pad_s)                              # and a second after the last
    nv = sampler.stop()
    nv['t_work_s'] = nv.t_s - work_start_s         # sampler time on the traffic clock
    return dict(steps=steps, idles=idles, nvml=nv, work_end_s=work_end_s,
                truncated=truncated, requests=pending, sched_report=sched.report())


def cumulative_energy(nv):
    '''(t, E) on the traffic clock. With an energy counter, keep only the moments it
    changed; without one, integrate board power.'''
    t = nv.t_work_s.to_numpy()
    if HAVE_ENERGY:
        e = nv.energy_j.to_numpy()
        keep = np.r_[True, np.diff(e) != 0]
        return t[keep], e[keep]
    w = nv.power_w.to_numpy()
    return t, np.r_[0.0, np.cumsum(0.5 * (w[1:] + w[:-1]) * np.diff(t))]


def energy_between(cum, a, b):
    '''Joules between traffic times a and b.'''
    t, e = cum
    return float(np.interp(b, t, e) - np.interp(a, t, e))

print('run loop defined')

In [ ]:
REAL_RUNS = {}
for name in RUN:
    REAL_RUNS[name] = []
    for k in range(REPEATS):
        t0 = time.time()
        run = run_real(make_traffic(name), sched_for(name), Executor())
        REAL_RUNS[name].append(run)
        nv, wall = run['nvml'], run['work_end_s']
        during = nv[(nv.t_work_s >= 0) & (nv.t_work_s <= wall)]
        status = 'TRUNCATED at MAX_RUN_S' if run['truncated'] else 'complete'
        n_steps = len(run['steps'])
        print(f'{name:7s} run {k + 1}: {n_steps:5d} steps in {wall:6.1f} s, '
              f'clock {during.sm_mhz.median():.0f} MHz (min {during.sm_mhz.min():.0f}), '
              f'{status}   [{time.time() - t0:.0f} s]')

## 8b · CPU time per step, fitted from the real runs

HuggingFace spends about 16 ms of CPU on every forward pass while the GPU mostly waits, so a real step lasts as long as the CPU takes. Every real step above was timed, so a straight-line fit gives that cost:

**step time ≈ ms per pass × passes + ms per token × tokens + ms per step**

With `HOST_TIME = True` the simulator uses it: a simulated step lasts the longer of its GPU kernel time and this CPU time, and the GPU waits out the difference at idle + `HOST_WAIT_W`. The gap between back-to-back steps (the scheduler's own time) is taken from the same runs. The constants are fitted on these runs, so the red line's timing is calibrated, not predicted: the energy comparison is unaffected.

In [ ]:
rows, gaps = [], []
for name in RUN:
    for run in REAL_RUNS[name]:
        st = run['steps']
        for i, s in enumerate(st):
            rows.append(dict(condition=name, step_ms=1000.0 * (s['t_end_s'] - s['t_start_s']),
                             passes=len(s['pieces']), tokens=sum(p[2] for p in s['pieces'])))
            if i + 1 < len(st):
                g = 1000.0 * (st[i + 1]['t_start_s'] - s['t_end_s'])
                if 0.0 <= g < 20.0:                      # back to back, no idle wait between
                    gaps.append(g)
HOSTDATA = pd.DataFrame(rows)
A = np.c_[HOSTDATA['passes'], HOSTDATA['tokens'], np.ones(len(HOSTDATA))]
coef, *_ = np.linalg.lstsq(A, HOSTDATA['step_ms'].to_numpy(), rcond=None)
per_pass, per_token, per_step = (max(0.0, float(c)) for c in coef)
fit = A @ np.array([per_pass, per_token, per_step])
resid = HOSTDATA['step_ms'].to_numpy() - fit
r2 = 1.0 - float((resid ** 2).sum() / ((HOSTDATA['step_ms'] - HOSTDATA['step_ms'].mean()) ** 2).sum())
GAP_MS_REAL = float(np.median(gaps)) if gaps else 0.05
HOST = (dict(host_ms_per_pass=per_pass, host_ms_per_token=per_token, host_ms_per_step=per_step,
             host_wait_w=HOST_WAIT_W, gap_ms=GAP_MS_REAL) if HOST_TIME else {})
print(f'real step time = {per_pass:.2f} ms per pass + {per_token:.4f} ms per token '
      f'+ {per_step:.2f} ms per step   (R^2 {r2:.3f} over {len(HOSTDATA)} steps)')
print(f'gap between back-to-back steps: {GAP_MS_REAL:.2f} ms (median of {len(gaps)})')
print('the simulator will use these' if HOST_TIME else 'HOST_TIME is off: kernel time only')
display(HOSTDATA.assign(fit_ms=fit).groupby('condition')[['step_ms', 'fit_ms']].median().round(2))

## 8c · The idle floor during each run

The last run showed the card idling 1-2 W **above** the idle measured once at the start, in the gaps between bursts, while the simulator drops to exactly idle there. So each run's floor is measured from its own idle gaps: the waits for the next request, each trimmed by 0.3 s so no step energy leaks in, used when they add up to at least 2 s. A run with fewer idle gaps (`bursty`, `prefill_bursty`) keeps the start floor.

The real line and the real energy above idle use this floor from here on. The summary also keeps the comparison against the start floor (`*_start_floor` columns), so both are visible.

In [ ]:
def in_run_idle_w(run, trim_s=0.3, min_gap_s=0.5, need_s=2.0):
    '''(watts, seconds used, source): board power while the card waited for the next request
    during this run, from its idle gaps; the start floor if they add up to under need_s.'''
    cum = cumulative_energy(run['nvml'])
    spans = [(a + trim_s, b) for a, b in run['idles'] if b - a >= min_gap_s + trim_s]
    secs = sum(b - a for a, b in spans)
    if secs < need_s:
        return IDLE_W, secs, 'start (too few idle gaps)'
    return sum(energy_between(cum, a, b) for a, b in spans) / secs, secs, 'idle gaps'

FLOOR, rows = {}, []
for name in RUN:
    FLOOR[name] = []
    for k, run in enumerate(REAL_RUNS[name]):
        w, secs, src = in_run_idle_w(run)
        FLOOR[name].append(w)
        rows.append(dict(condition=name, run=k + 1, start_idle_w=IDLE_W, in_run_idle_w=w,
                         above_start_w=w - IDLE_W, idle_gap_s=secs, floor_from=src))
FLOORS = pd.DataFrame(rows)
display(FLOORS.round(2))

## 9 · Simulate each condition, and price the real steps

For each condition this builds three traces, all ordinary `EngineTrace` objects, so they share one CSV format:

- **sim**: `run_engine` on the same traffic, its own clock.
- **real**: one row per real step. Times are measured, energy comes from NVML, and batch, token, KV and FLOP columns are computed from the pieces that ran, exactly the way the simulator computes them.
- **sim, same steps**: the real steps again, with energy = this card's idle floor + the simulator's predicted dynamic energy for those exact pieces (EnergAIzer, with the extra kernels).

**Energy above idle** is what gets compared. Each side subtracts its own floor (this card's measured idle for the real run, EnergAIzer's 47.35 W for its kernels), so the SXM-vs-PCIe idle difference drops out. Board power is in the summary too.

In [ ]:
from dynshape import (EngineConfig, EngineTrace, IterationRecord, Segment, run_engine,
                      kernel_work, WORK_FIELDS, iteration_token_shapes)
from dynshape.engine import price_kernel

OUT = 'compare_out'
os.makedirs(OUT, exist_ok=True)
SKIPPED = collections.Counter()

def expand(pieces, exact):
    '''Recorded pieces -> (Piece objects, tagged kernels), one forward pass per piece.'''
    ps = [Piece(kind=k, request_id=r, tokens=t, context=c) for k, r, t, c in pieces]
    with (exact_contexts() if exact else contextlib.nullcontext()):
        return ps, build_iteration_kernels_tagged(rw, ps, fuse_linear=False, extras=XS)

def work_vector(ps, tagged):
    '''Batch columns and FLOPs/bytes -- the same accounting execute_iteration uses.'''
    sh = iteration_token_shapes(ps)
    share = sh['prefill_tokens'] / sh['total_tokens'] if sh['total_tokens'] else 0.0
    w = {f: 0.0 for f in WORK_FIELDS}
    for (q, op), tag in tagged:
        phase = tag.partition('+')[0]          # an extra kernel is tagged '<phase>+<name>'
        fl, by, wb = kernel_work(q, op, strict=False)
        if phase.endswith('prefill') or phase.endswith('decode'):
            w['attn_flops'] += fl
            w['attn_bytes'] += by
            w['prefill_flops' if phase.endswith('prefill') else 'decode_flops'] += fl
        else:
            w['linear_flops'] += fl
            w['linear_bytes'] += by
            w['weight_bytes'] += wb
            w['prefill_flops'] += fl * share
            w['decode_flops'] += fl * (1.0 - share)
    return sh, w

def predicted_dynamic(tagged):
    '''(dynamic J, kernel ms): energy above the idle floor, priced exactly as run_engine
    prices it -- measured prices for the extras that have one, EnergAIzer for the rest.'''
    t_ms = e_j = 0.0
    for (q, op), tag in tagged:
        try:
            kt, _kp, ke = price_kernel(gee, q, op, tag, XS, LUT_IDLE_W)
        except Exception:
            SKIPPED[op[0]] += 1
            continue
        t_ms += kt
        e_j += ke
    return e_j - LUT_IDLE_W * t_ms / 1000.0, t_ms


def build_traces(name, run):
    '''Two EngineTraces on the REAL timeline -- what NVML measured, and what the simulator
    predicts for the very same steps -- plus one row per step for the step-time plot.'''
    nv, end = run['nvml'], run['work_end_s']
    cum = cumulative_energy(nv)
    events = sorted([(s['t_start_s'], 'step', s) for s in run['steps']]
                    + [(a, 'idle', None) for a, _b in run['idles']], key=lambda ev: ev[0])
    starts = [ev[0] for ev in events] + [end]
    cfg = EngineConfig(scheduler=sched_for(name), fuse_linear=False, idle_w=IDLE_W, extras=XS)
    real = EngineTrace(requests=list(run['requests']), scheduler_report=run['sched_report'],
                       config=cfg)
    pred = EngineTrace(requests=list(run['requests']), scheduler_report=run['sched_report'],
                       config=cfg)
    rows = []

    def add_idle(a, b):
        e = energy_between(cum, a, b)
        real.idle_segments.append(Segment(kind='IDLE', t_start_ms=1000 * a, time_ms=1000 * (b - a),
                                          power_w=e / (b - a), energy_j=e, op='idle'))
        pred.idle_segments.append(Segment(kind='IDLE', t_start_ms=1000 * a, time_ms=1000 * (b - a),
                                          power_w=IDLE_W, energy_j=IDLE_W * (b - a), op='idle'))

    if starts[0] > 0:
        add_idle(0.0, starts[0])
    # Every moment belongs to exactly one event: a step (its GPU span, then the gap until
    # the next event) or an idle wait. So the energies add up to the whole run.
    for i, (a, kind, s) in enumerate(events):
        b = starts[i + 1]
        if b <= a:
            continue
        if kind == 'idle':
            add_idle(a, b)
            continue
        dur = s['t_end_s'] - a
        span = max(b - a, dur)
        gap = span - dur
        ps, exact_kernels = expand(s['pieces'], exact=True)
        sh, work = work_vector(ps, exact_kernels)
        _, sim_kernels = expand(s['pieces'], exact=False)
        dyn_j, kern_ms = predicted_dynamic(sim_kernels)
        e_real = energy_between(cum, a, a + span)
        e_pred = IDLE_W * span + dyn_j
        inside = nv.power_w[(nv.t_work_s >= a) & (nv.t_work_s <= a + span)]
        common = dict(t_start_ms=1000 * a, duration_ms=1000 * dur, gap_ms=1000 * gap,
                      decode_batch=sh['decode_batch'], prefill_chunks=sh['prefill_chunks'],
                      prefill_tokens=sh['prefill_tokens'], decode_tokens=sh['decode_batch'],
                      total_tokens=sh['total_tokens'], context_mean=sh['context_mean'],
                      context_max=sh['context_max'], n_kernels=len(exact_kernels), n_skipped=0,
                      energy_fused_j=np.nan, energy_attn_prefill_j=np.nan,
                      energy_attn_decode_j=np.nan,
                      n_waiting=s['n_waiting'], n_running=s['n_running'],
                      kv_blocks=s['kv_blocks'], kv_utilisation=s['kv_utilisation'],
                      preemptions_total=s['preemptions_total'], **work)
        real.iterations.append(IterationRecord(
            idx=len(real.iterations), energy_j=e_real, avg_power_w=e_real / span,
            peak_power_w=float(inside.max()) if len(inside) else e_real / span, **common))
        pred.iterations.append(IterationRecord(
            idx=len(pred.iterations), energy_j=e_pred, avg_power_w=e_pred / span,
            peak_power_w=np.nan, **common))
        rows.append(dict(real_ms=1000 * dur, sim_kernel_ms=kern_ms, passes=len(ps),
                         prefill_tokens=sh['prefill_tokens']))
    return real, pred, rows


def dyn_energy(tr, floor):
    '''Energy above an idle floor, over the whole trace.'''
    return tr.total_energy_j - floor * tr.total_time_ms / 1000.0


SIM, REAL_TR, PRED_TR, STEPS = {}, {}, {}, []
for name in RUN:
    t0, shapes0 = time.time(), len(gee.cache)
    SIM[name] = run_engine(make_traffic(name), rw, gee,
                           EngineConfig(scheduler=sched_for(name), fuse_linear=False,
                                        record_kernels_until_ms=0.0, idle_w=LUT_IDLE_W, model=MC,
                                        extras=XS, **HOST))
    REAL_TR[name], PRED_TR[name] = [], []
    for k, run in enumerate(REAL_RUNS[name]):
        real, pred, rows = build_traces(name, run)
        REAL_TR[name].append(real)
        PRED_TR[name].append(pred)
        STEPS += [dict(condition=name, run=k + 1, **r) for r in rows]
    new_shapes = len(gee.cache) - shapes0
    print(f'{name:14s} simulated and priced in {time.time() - t0:.0f} s ({new_shapes:,} new shapes)')
STEPS = pd.DataFrame(STEPS)
skipped_msg = dict(SKIPPED) or 'none'
print(f'kernels EnergAIzer could not price: {skipped_msg}')
if XS is not None:
    print('simulator:', SIM[RUN[0]].shape_report)

rows = []
for name in RUN:
    sim = SIM[name]
    ss = sim.summary()
    sim_wall = sim.total_time_ms / 1000.0
    sim_dyn = dyn_energy(sim, LUT_IDLE_W)
    for k, (real, pred) in enumerate(zip(REAL_TR[name], PRED_TR[name])):
        sr = real.summary()
        wall = real.total_time_ms / 1000.0
        floor = FLOOR[name][k]
        real_dyn = dyn_energy(real, floor)              # above the idle floor measured in this run
        real_dyn_start = dyn_energy(real, IDLE_W)       # above the idle measured once at the start
        rows.append({
            'condition': name, 'run': k + 1, 'requests': sr['requests'],
            'idle_w_start': IDLE_W, 'idle_w_in_run': floor, 'dyn_j_real_start_floor': real_dyn_start,
            'steps_real': sr['iterations'], 'steps_sim': ss['iterations'],
            'wall_s_real': wall, 'wall_s_sim': sim_wall,
            'dyn_j_real': real_dyn, 'dyn_j_sim_same_steps': dyn_energy(pred, IDLE_W),
            'dyn_j_sim': sim_dyn,
            'avg_dyn_w_real': real_dyn / wall, 'avg_dyn_w_sim': sim_dyn / sim_wall,
            'board_w_real': sr['avg_power_w_wallclock'], 'board_w_sim': ss['avg_power_w_wallclock'],
            'tokens_per_s_real': sr['output_tokens_per_s'],
            'tokens_per_s_sim': ss['output_tokens_per_s'],
            'ttft_p50_s_real': sr['ttft_p50_s'], 'ttft_p50_s_sim': ss['ttft_p50_s'],
            'itl_p50_ms_real': sr['itl_p50_ms'], 'itl_p50_ms_sim': ss['itl_p50_ms']})
SUMMARY = pd.DataFrame(rows)
SUMMARY['wall_ratio_sim_over_real'] = SUMMARY.wall_s_sim / SUMMARY.wall_s_real
SUMMARY['dyn_err_same_steps_pct'] = 100 * (SUMMARY.dyn_j_sim_same_steps / SUMMARY.dyn_j_real - 1)
SUMMARY['dyn_err_sim_pct'] = 100 * (SUMMARY.dyn_j_sim / SUMMARY.dyn_j_real - 1)
SUMMARY['dyn_err_same_steps_start_floor_pct'] = 100 * (SUMMARY.dyn_j_sim_same_steps / SUMMARY.dyn_j_real_start_floor - 1)
SUMMARY['dyn_err_sim_start_floor_pct'] = 100 * (SUMMARY.dyn_j_sim / SUMMARY.dyn_j_real_start_floor - 1)
display(SUMMARY.set_index(['condition', 'run']).T.round(3))

## 10 · Plots

Each condition gets two panels on one time axis:

- **power above idle**: black is the real A100, yellow is the simulator on its own clock. If yellow ends early, the simulator is missing *time*, not necessarily energy.
- **traffic** underneath: the dotted line counts requests arriving in each window; the bars count forward passes run in that window on the real GPU, prefill in purple and decode in green.

The real line is power above the idle floor measured during its own run (section 8c), the simulator above its own floor. Board power is this plus about 48-51 W. The simulator-on-the-real-steps view is in the summary table (`*_same_steps` columns), no longer plotted.

In [ ]:
SIM_COLOR = '#FFB300'                      # bright amber-yellow: the simulator, own clock
PREFILL_C, DECODE_C = '#6a3d9a', '#1b9e77'  # traffic bars: prefill purple, decode green


def dt_for(name):
    '''The plot and comparison window of a condition.'''
    return CONDITIONS[name].get('dt', PLOT_DT_S)


def dyn_series(tr, floor, total_s, dt_s):
    '''Power above an idle floor on a fixed grid, padded at the floor past the trace end.'''
    t_ms, p = tr.resample(dt_ms=1000 * dt_s, total_ms=1000 * total_s, pad_power_w=floor)
    return t_ms / 1000.0, p - floor


def traffic_series(run, total_s, dt_s):
    '''Per window: requests arriving, and the prefill and decode passes the real GPU ran.'''
    n = int(np.ceil(total_s / dt_s)) + 1
    arrivals, prefill, decode = np.zeros(n), np.zeros(n), np.zeros(n)
    for r in run['requests']:
        i = int(r.arrived_at // dt_s)
        if 0 <= i < n:
            arrivals[i] += 1
    for s in run['steps']:
        i = int(s['t_start_s'] // dt_s)
        if 0 <= i < n:
            for kind, _rid, _tokens, _ctx in s['pieces']:
                if kind == 'prefill':
                    prefill[i] += 1
                else:
                    decode[i] += 1
    return np.arange(n) * dt_s, arrivals, prefill, decode


SERIES, CORR = {}, {}
GROUPS = {}
for name in RUN:
    GROUPS.setdefault(CONDITIONS[name].get('scale', 'base'), []).append(name)

for tag, names in GROUPS.items():              # one figure per length: short, long
    fig = plt.figure(figsize=(13, 4.6 * len(names)))
    gs = fig.add_gridspec(2 * len(names), 1, height_ratios=[3.0, 1.3] * len(names), hspace=0.6)
    for j, name in enumerate(names):
        dt = dt_for(name)
        ax = fig.add_subplot(gs[2 * j])
        axt = fig.add_subplot(gs[2 * j + 1], sharex=ax)
        real, pred, sim = REAL_TR[name][0], PRED_TR[name][0], SIM[name]
        total = max(real.total_time_ms, sim.total_time_ms) / 1000.0
        t, p_real = dyn_series(real, FLOOR[name][0], total, dt)    # above this run's own idle floor
        _, p_pred = dyn_series(pred, IDLE_W, total, dt)
        _, p_sim = dyn_series(sim, LUT_IDLE_W, total, dt)
        n = min(len(t), len(p_pred), len(p_sim))
        SERIES[name] = pd.DataFrame({'t_s': t[:n], 'real_dyn_w': p_real[:n],
                                     'sim_same_steps_dyn_w': p_pred[:n], 'sim_dyn_w': p_sim[:n]})
        ser = SERIES[name]
        m = (ser.t_s < real.total_time_ms / 1000.0).to_numpy()
        CORR[name] = (float(np.corrcoef(ser.real_dyn_w[m], ser.sim_dyn_w[m])[0, 1])
                      if m.sum() > 2 else np.nan)
        ax.step(ser.t_s, ser.sim_dyn_w, where='post', color=SIM_COLOR, lw=2.6,
                label='simulator, its own clock')
        ax.step(ser.t_s, ser.real_dyn_w, where='post', color='black', lw=1.3,
                label='real A100 (NVML)')
        s = SUMMARY[(SUMMARY.condition == name) & (SUMMARY.run == 1)].iloc[0]
        ax.set_title(f'{name} ({dt:g} s windows): real {s.wall_s_real:.1f} s vs simulated '
                     f'{s.wall_s_sim:.1f} s   |   energy above idle: real {s.dyn_j_real:.0f} J, '
                     f'simulator {s.dyn_j_sim:.0f} J ({s.dyn_err_sim_pct:+.0f}%)   |   '
                     f'r = {CORR[name]:.2f}', fontsize=10)
        ax.set_ylabel('power above idle (W)')
        ax.grid(alpha=.25)
        ax.legend(fontsize=13, loc='upper right')
        plt.setp(ax.get_xticklabels(), visible=False)

        tt, arrivals, prefill, decode = traffic_series(REAL_RUNS[name][0], total, dt)
        arrivals, prefill, decode = arrivals / dt, prefill / dt, decode / dt   # per second
        axt.bar(tt, prefill, width=dt, align='edge', color=PREFILL_C, label='prefill passes')
        axt.bar(tt, decode, width=dt, align='edge', bottom=prefill, color=DECODE_C,
                label='decode passes')
        axt.step(tt, arrivals, where='post', color='black', lw=1.0, ls=':', label='requests arriving')
        axt.set_ylabel('forward passes/s (bars),\nrequests arriving/s (dotted)', fontsize=8)
        axt.set_xlabel('time from first request (s)', fontsize=8)
        axt.grid(alpha=.25)
        axt.legend(fontsize=7, loc='upper right', ncol=3)
    fig.savefig(f'{OUT}/fig_power_traces' + ('' if tag == 'base' else f'_{tag}') + '.png', dpi=120, bbox_inches='tight')
    plt.show()

In [ ]:
LABEL = {'real': 'real A100', 'sim': 'simulator, own clock'}
COLOR = {'real': 'black', 'sim': SIM_COLOR}
S1 = SUMMARY[SUMMARY.run == 1].set_index('condition').loc[RUN]
panels = [('time the run took (s)', [('real', 'wall_s_real'), ('sim', 'wall_s_sim')], False),
          ('energy above idle (J, log scale)', [('real', 'dyn_j_real'), ('sim', 'dyn_j_sim')], True),
          ('average power above idle (W)', [('real', 'avg_dyn_w_real'), ('sim', 'avg_dyn_w_sim')], False)]

fig, axes = plt.subplots(1, 3, figsize=(17, 4.4))
x = np.arange(len(RUN))
for ax, (title, bars, logy) in zip(axes, panels):
    w = 0.8 / len(bars)
    for j, (who, col) in enumerate(bars):
        ax.bar(x + (j - (len(bars) - 1) / 2) * w, S1[col].to_numpy(), w,
               color=COLOR[who], edgecolor='#555555', linewidth=0.4, label=LABEL[who])
    if logy:
        ax.set_yscale('log')
    ax.set_xticks(x)
    ax.set_xticklabels(RUN, rotation=30, ha='right', fontsize=8)
    ax.set_title(title)
    ax.grid(alpha=.25, axis='y')
    ax.legend(fontsize=8)
plt.tight_layout()
fig.savefig(f'{OUT}/fig_summary_bars.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
palette = ['#2ca02c', '#1f77b4', '#ff7f0e', '#9467bd', '#d62728', '#8c564b',
           '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']
for name, c in zip(RUN, palette * 3):
    g = STEPS[(STEPS.condition == name) & (STEPS.run == 1)]
    ax[0].scatter(g.sim_kernel_ms, g.real_ms, s=7, alpha=.5, color=c, label=name)
    ax[1].scatter(g.passes, g.real_ms - g.sim_kernel_ms, s=7, alpha=.5, color=c, label=name)
lim = [0, max(STEPS.real_ms.max(), STEPS.sim_kernel_ms.max())]
ax[0].plot(lim, lim, 'k--', lw=.8, label='equal')
ax[0].set_xlabel('simulator: kernel time for the step (ms)')
ax[0].set_ylabel('real: measured step time (ms)')
ax[0].set_title('step time, same steps')
ax[1].set_xlabel('forward passes in the step')
ax[1].set_ylabel('real minus simulated (ms)')
ax[1].set_title('time the simulator does not model')
for a in ax:
    a.grid(alpha=.25)
    a.legend(fontsize=8)
plt.tight_layout()
fig.savefig(f'{OUT}/fig_step_time.png', dpi=150, bbox_inches='tight')
plt.show()

per_pass = (STEPS.real_ms - STEPS.sim_kernel_ms) / STEPS.passes
OVERHEAD = STEPS.assign(per_pass=per_pass).groupby('condition').per_pass.median().loc[RUN]
for name, v in OVERHEAD.items():
    print(f'{name:14s} real minus simulated time per forward pass: {v:.2f} ms (median)')

## 11 · Export

| file | what it is |
|---|---|
| `real_CONDITION_iterations.csv` | one row per real step, same 34 columns as `Bucketing_measured_iterations.csv` |
| `sim_CONDITION_iterations.csv` | one row per simulated step, same 34 columns |
| `sim_same_steps_CONDITION_iterations.csv` | the real steps priced by the simulator, same 34 columns |
| `real_/sim_CONDITION_requests.csv` | per request: arrival, TTFT, inter-token latency |
| `power_1000ms_CONDITION.csv` | the plotted series: power above idle, both lines |
| `nvml_CONDITION.csv` | raw NVML samples (power, energy counter, clock) |
| `summary.csv`, `steps_real_vs_sim.csv`, `opcheck.csv`, `ktotals.csv`, `pass_check.csv`, `idle_floors.csv` | the tables above |
| `fig_*.png` | the three figures |

With `REPEATS` above 1, real files get a `_runN` suffix.

In [ ]:
import zipfile

def sfx(k):
    return '' if REPEATS == 1 else f'_run{k + 1}'

for name in RUN:
    sim_iters, sim_reqs, _ = SIM[name].to_dataframes()
    sim_iters.to_csv(f'{OUT}/sim_{name}_iterations.csv', index=False)
    sim_reqs.to_csv(f'{OUT}/sim_{name}_requests.csv', index=False)
    for k, (real, pred) in enumerate(zip(REAL_TR[name], PRED_TR[name])):
        real_iters, real_reqs, _ = real.to_dataframes()
        real_iters.to_csv(f'{OUT}/real_{name}{sfx(k)}_iterations.csv', index=False)
        real_reqs.to_csv(f'{OUT}/real_{name}{sfx(k)}_requests.csv', index=False)
        pred.to_dataframes()[0].to_csv(f'{OUT}/sim_same_steps_{name}{sfx(k)}_iterations.csv',
                                       index=False)
        REAL_RUNS[name][k]['nvml'].to_csv(f'{OUT}/nvml_{name}{sfx(k)}.csv', index=False)
    SERIES[name].to_csv(f'{OUT}/power_{int(dt_for(name) * 1000)}ms_{name}.csv', index=False)

SUMMARY.to_csv(f'{OUT}/summary.csv', index=False)
STEPS.to_csv(f'{OUT}/steps_real_vs_sim.csv', index=False)
TRAFFIC.to_csv(f'{OUT}/traffic.csv')
for tname in ('OPCHECK', 'KTOTALS'):
    if tname in globals():
        globals()[tname].to_csv(f'{OUT}/{tname.lower()}.csv', index=False)
if XS is not None:
    XS.save(f'{OUT}/extras.json')                       # with the measured prices, if any
if len(BENCH):
    BENCH.to_csv(f'{OUT}/measured_prices.csv', index=False)
if len(PASSCHECK):
    PASSCHECK.to_csv(f'{OUT}/pass_check.csv')
FLOORS.to_csv(f'{OUT}/idle_floors.csv', index=False)
pd.DataFrame([{'conditions': ','.join(RUN), 'repeats': REPEATS, 'bucket': BUCKET,
               'extras': XS.name if XS is not None else 'off',
               'measured_prices': ','.join(XS.priced) if XS is not None else '',
               'model': MODEL_ID, 'branch': BRANCH, 'target_mhz': TARGET_MHZ, 'card': CARD, 'idle_w': IDLE_W,
               'lut_idle_w': LUT_IDLE_W, 'plot_dt_s': PLOT_DT_S}]).to_csv(
    f'{OUT}/settings.csv', index=False)

with zipfile.ZipFile('qwen3_sim_vs_real.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(os.listdir(OUT)):
        z.write(os.path.join(OUT, f), f)
print(f'qwen3_sim_vs_real.zip: {len(os.listdir(OUT))} files')
for f in sorted(os.listdir(OUT)):
    print('  ', f)

try:
    from google.colab import files
    files.download('qwen3_sim_vs_real.zip')
except Exception:
    print('(not on Colab)')

## Notes

- **The kernel list is a prediction.** It is written from transformers' source for eager `Qwen3ForCausalLM`; section 5 checks it against the profiler. Send the section 5 output if any op differs.
- **No measured prices yet.** For GPT-2, EnergAIzer priced the LM head 3.5x too fast. Qwen3's LM head is 151,936 wide and its GEMMs are much bigger than anything EnergAIzer's tables were built from, so section 5's `real_gpu_ms` vs `energaizer_ms` is the first place to look if the energy is off.
- **The power signal is large now.** A 14B model draws tens to hundreds of watts above idle, against GPT-2's 3-4 W, so the idle-floor noise that blurred the GPT-2 decode results should matter much less.
- **Same executor caveats as GPT-2.** HuggingFace eager, one forward pass per request, so the CPU time per pass is large and the batch never fuses; production engines (vLLM, TensorRT-LLM) would look different.
- **One card, one clock:** A100-SXM4-40GB at 900 MHz.